# M04 — Nested validation and the limits of site adjustment

<!-- paper-first -->
### Research question

**Reading:** [PM03](../../curriculum/papers/modeling.md#pm03), [PD02](../../curriculum/papers/design.md#pd02). Review the assigned figure or result before starting the lesson.

**Question:** What would count as a genuinely new person, site, or population for this evaluation?

Record a prediction, a source location, and one point you want this lesson to clarify. Ask your AI tutor to distinguish the paper’s evidence from its interpretation.
<!-- /paper-first -->

**Original guided lab · 75–100 minutes.** Read 20 min, predict/code 35 min, failure investigation 20 min, explain and transfer 15 min. Run all cells in order in a fresh kernel. All executed data are synthetic unless explicitly stated. No network, GPU, or external dataset is required.

Validation design begins with the claim, not with a library default. A new scan from a person already represented in training differs from a new person; a new person at a familiar site differs from a new hospital. Repeated scans and related people require grouping. Sites add another dependency and another possible change in measurement. Record the intended deployment unit before fitting anything.

Nested validation separates choosing a model from estimating the performance of that choice. Inner folds select hyperparameters using only an outer training portion. The selected pipeline is then evaluated on its untouched outer test portion. Scaling, feature selection, dimensionality reduction, and fitted nuisance adjustments must be repeated inside the relevant folds. A pipeline automates fitting order but cannot repair input features already selected using the full dataset.

The first experiment uses repeated rows from each synthetic person. Outer and inner GroupKFold both receive person IDs. Assertions explicitly inspect the sets, so the learner can see what grouping protects. Outer fold estimates are not independent clinical trials; their training sets overlap. Reporting their spread is useful description but does not automatically justify a simple independent-observation confidence interval.

Site harmonization attempts to reduce unwanted measurement differences while retaining specified biological variation. That objective requires assumptions and information. If every patient is at one site and every control at another, site and group are perfectly confounded: the data cannot tell which produced the difference. Blindly removing the site mean can remove the entire target signal. Our second experiment constructs exactly this non-identifiability; the operation runs correctly while answering an impossible separation question.

ComBat and related methods are more elaborate than subtracting a site average. The original neuroHarmonize documentation distinguishes learning harmonization parameters from applying them to separate data and provides covariate controls. This notebook does not implement ComBat. It illustrates why fitting, covariate choice, and target-setting assumptions must be audited. An unseen site may require calibration data or a method explicitly designed for that setting; knowing how to transform held-out people at a known site does not automatically solve unseen-site generalization.

Ask the AI to list what information will be available for a future individual. If its adjustment uses that person’s future outcome or a mean computed from a forbidden test cohort, the evaluation no longer matches the intended prediction scenario. A valid result sometimes requires stating that the desired claim is unsupported by the available design.

## Transformation contract

Participant IDs and feature/target rows → nested disjoint-person folds → training-only pipeline fits → outer MAE. Separate demonstration: site-constant data → site demeaning → destroyed target contrast. The latter is an illustration, not ComBat.

## Ask your AI tutor

```text
Explain this notebook one transformation at a time.
Before each cell ask me to predict shapes, units, and a check.
Give edits in executable cells of at most 20 lines.
Keep the prescribed split, random seed, and tests intact.
Distinguish generated suggestions from executed results.
After the failure experiment, ask me to explain the mechanism.
```

In [1]:
import numpy as np
from sklearn.model_selection import GroupKFold,GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
rng=np.random.default_rng(404)
groups=np.repeat(np.arange(60),2)
base=rng.normal(size=(60,8));X=np.repeat(base,2,axis=0)
y=np.repeat(2*base[:,0]+rng.normal(0,.4,60),2)
errors=[]
for tr,te in GroupKFold(4).split(X,y,groups):
    assert not set(groups[tr])&set(groups[te])
    inner=GroupKFold(3)
    for a,b in inner.split(X[tr],y[tr],groups[tr]):assert not set(groups[tr][a])&set(groups[tr][b])
    search=GridSearchCV(make_pipeline(StandardScaler(),Ridge()),{'ridge__alpha':[.1,1,10]},cv=inner)
    search.fit(X[tr],y[tr],groups=groups[tr])
    errors.append(mean_absolute_error(y[te],search.predict(X[te])))
print('Outer-fold MAE:',errors)
assert len(errors)==4 and np.mean(errors)<1


Outer-fold MAE: [0.5010350531826453, 0.33817584254808164, 0.2594385653761417, 0.4023449679823005]


In [2]:
site=np.repeat([0,1],20);diagnosis=site.copy()
feature=2*site.astype(float)
adjusted=feature.copy()
for s in [0,1]:adjusted[site==s]-=feature[site==s].mean()
design=np.column_stack([np.ones(40),site,diagnosis])
print('Rank / columns:',np.linalg.matrix_rank(design),design.shape[1])
print('Before/after group difference:',feature[diagnosis==1].mean()-feature[diagnosis==0].mean(),np.ptp(adjusted))
assert np.linalg.matrix_rank(design)==2 and np.all(adjusted==0)


Rank / columns: 2 3
Before/after group difference: 2.0 0.0


**A different claim: unseen-site evaluation.** Assign each participant to one of three toy sites, introduce a site-dependent feature offset, and hold out entire sites. Both site and participant intersections must be empty. This evaluates a domain shift; it does not fit or validate harmonization.

In [3]:
site_id=np.repeat(np.arange(60)%3,2)
X_site=X.copy();X_site[:,0]+=2*site_id
site_errors=[]
for held_site in np.unique(site_id):
    train_site=np.flatnonzero(site_id!=held_site);test_site=np.flatnonzero(site_id==held_site)
    assert not set(site_id[train_site])&set(site_id[test_site])
    assert not set(groups[train_site])&set(groups[test_site])
    estimator=make_pipeline(StandardScaler(),Ridge(alpha=1)).fit(X_site[train_site],y[train_site])
    site_errors.append(mean_absolute_error(y[test_site],estimator.predict(X_site[test_site])))
print('Unseen-site MAE with a measurement offset:',site_errors)
assert len(site_errors)==3 and np.all(np.isfinite(site_errors))


Unseen-site MAE with a measurement offset: [1.6708239785557724, 1.2386981804367234, 2.72289514378277]


## Deliberate failure and repair

Site demeaning erases a perfectly site-confounded target. There is no algebraic repair that recovers separately identifiable site and diagnosis effects from this design alone. Repair the study with overlap, additional information, or a narrower claim. Do not claim that nested folds make unidentifiable biology identifiable.

## Your investigation

Draw outer person folds, inner person folds, and a separate leave-site-out design. Specify whether any person appears at multiple sites and how both restrictions will be satisfied. Write a harmonization contract: retained covariates, fitted rows, future inputs, unseen-site policy, and diagnostic plots.

## Transfer to real neuroimaging

Move to neuroHarmonize only after specifying a supported training/application setting and evaluating preservation of intended effects. Real data need site and participant identifiers, overlap checks, feature provenance, and independent assessment. No ComBat package or unseen-site adaptation is executed here.

**Primary teaching sources, pinned where hosted on GitHub:**

- [BrainIAK: nested validation and circular inference](https://github.com/brainiak/brainiak-tutorials/blob/fb62ede943d9694fe703aee0df5f43ecf5558415/tutorials/05-classifier-optimization.ipynb)
- [neuroHarmonize official repository](https://github.com/rpomponio/neuroHarmonize)

Pinned upstream tutorials are a separate assignment; they have **not been executed** by this core lab. They may require data downloads, specialist dependencies, unfinished student cells, and additional compute.

## Exit questions and answer key

1. Which score evaluates hyperparameter selection? **Outer test scores, provided they are not reused to redesign the method.**
2. Can ComBat identify perfectly confounded site and diagnosis effects without further information? **No general adjustment removes the underlying non-identifiability.**

### Return to the research question

Revisit [PM03](../../curriculum/papers/modeling.md#pm03), [PD02](../../curriculum/papers/design.md#pd02) and your initial prediction. In your [evidence ledger](../../curriculum/coursework/EVIDENCE_LEDGER.md):

1. Cite one output or diagnostic from this lesson and explain the transformation it demonstrates.
2. Revise one claim or question from the paper, with a figure or section locator. Which part of the published result remains open after this exercise?
3. Ask AI to propose a next check. Accept, revise or reject it with a scientific reason. Then explain your decision aloud without reading the AI response.

Include this entry in the A2 portfolio when relevant.
